# Phase3: 文字認識（OCR） — 「例から読み方を学ぶAI」

> ⏱ 所要時間: 説明を読む 25分 ＋ 実行 約3分

## このプロジェクトのゴール（おさらい）
駐車場のナンバー読み取り精算システムのミニ版を作り、**AIはシステムの一部品**であることを確かめます。

| Phase | やったこと | 分かったこと |
|---|---|---|
| 1 | 機械学習なしで、板を探し・読む | きれいな条件ではほぼ完璧、条件が悪いと急に壊れる |
| 2 | 板の**四隅**を学習するAI（検出） | 検出はほぼ100%成功。ただし**読み取りには四隅が1〜2px以内**の精度が必要。Hard は**検出が成功しても読めない** |

## このNotebookのゴール
Phase1 の**テンプレートマッチ**（人間が「文字の位置」と「見本」を書いた方法）を、**例から学ぶAI（CRNN＋CTC）**に置き換えます。次の3つを確かめます。
1. 学習するOCRは、**劣化した画像（Medium）**や**四隅のずれ**に強いか
2. **Hard**（人間にも読みにくい画像）は、どこまで読めるか。読めないとき、何が起きるか
3. 学習のしかた（**四隅のずれをどれだけ学習に混ぜるか**）で、結果はどう変わるか

## キーワード
| 用語 | 意味 |
|---|---|
| **OCR** | 画像の中の文字を、文字列として読み取ること |
| **CNN** | 画像から特徴（線・形）を取り出すAI部品 |
| **RNN／LSTM** | 並び（系列）を、前後の文脈を見ながら処理するAI部品。ここでは、板を**左から右へ読む**ために使う |
| **CRNN** | CNN で特徴を取り、RNN で並びを読む組み合わせ。文字認識の定番 |
| **CTC** | 「画像のどこに、どの文字があるか」の**位置合わせなしで**、文字列を学習する仕組み |
| **blank（空白記号）** | CTCの「ここには何もない」という特別な記号。同じ文字が続くときの区切りにも使う |
| **貪欲デコード（greedy）** | 各位置でいちばん確からしい記号を選び、連続をまとめ、blankを除いて文字列にする方法 |
| **完全一致／CER** | 文字列が全部正解の割合／文字の間違いの割合 |
| **確信度** | AIが「どれくらい自信があるか」。ここでは文字ごとの確率を使う |

In [ ]:
import sys
from pathlib import Path

# 正しい環境（プロジェクトの .venv）で動いているか確認する
try:
    import torch, PIL, numpy, pandas  # noqa: F401
except ModuleNotFoundError:
    raise SystemExit(
        f"必要なライブラリが見つかりません。今のPython: {sys.executable}\n"
        "→ カーネルを「Python (plate_reader_lab)」（または プロジェクトの .venv/bin/python）に切り替えてください。"
    )

# src/ をimportできるようにする（notebooks/ からでもプロジェクト直下からでも動く）
sys.path.insert(0, str(next(p for p in [Path.cwd() / "src", Path.cwd().parent / "src"] if p.exists())))

import time
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from torch import nn
from torch.utils.data import DataLoader

from plate_synth import generate_sample, split_seed
from ai_ocr.data import OcrSynthDataset, collate, jitter_corners, plate_to_tensor
from ai_ocr.evaluate import evaluate, pretty, summarize
from ai_ocr.model import BLANK, CHARS, CRNN, VOCAB, decode_greedy, encode
from classical import read_text, rectify
from plate_eval import exact_match

plt.rcParams["font.family"] = ["Hiragino Sans", "AppleGothic", "DejaVu Sans"]
%matplotlib inline
ROOT = next(p for p in [Path.cwd(), Path.cwd().parent] if (p / "src").exists())
device = "mps" if torch.backends.mps.is_available() else "cpu"
print("使用デバイス:", device)

上のセルは、道具の読み込みです。エラーが出たら、右上のカーネルを **Python (plate_reader_lab)** に切り替えてください。

> 📌 このNotebookの一部（「本実験の結果」「読み取りの観察」）は、本実験で学習済みのモデル（`outputs/ocr/`）を使います。**まだなら、ターミナルで `bash src/run_phase2.sh`（検出器、約50分）と `bash src/run_phase3.sh`（OCR、約60分）**を実行してください。モデルがない場合は、その旨が表示されます。

## 1. 入力を作る — 板を「1行」にする

OCRの入力は、**四隅で板をまっすぐに直した画像**です（Phase1 と同じ）。CRNNは「1行の画像」を左から右へ読むので、2段の板を次のように**1行に並べ替え**ます。

```
 [上段: 地名・分類番号] ＋ [下段: かな・一連番号]   →   1行の画像（読む順に、左から右へ）
```

**ポイント**: 上段 → 下段の順に横につなぐと、**文字列の順序（地名→分類番号→かな→一連番号）が、そのまま左から右の並び**になります。
（他の案: 上下を別々に読む、など。ここでは最も簡単な方法を採りました。）

In [ ]:
s = generate_sample(split_seed("val", "medium", 5), "medium")
plate = rectify(s.image, s.label["corners"])                  # 四隅で板をまっすぐにする（384×192）
x = plate_to_tensor(plate)                                    # 上段・下段に分け、横につなぐ → (3, 48, 512)
print("正解:", s.label["text"], "／ モデルへの入力の形:", tuple(x.shape))

fig, ax = plt.subplots(2, 1, figsize=(9, 4.4), gridspec_kw={"height_ratios": [2, 1]})
ax[0].imshow(plate); ax[0].set_title("まっすぐに直した板（384×192）"); ax[0].axis("off")
ax[1].imshow(x.permute(1, 2, 0).numpy()); ax[1].set_title("モデルへの入力: 上段（左半分）＋下段（右半分）を横につなげた1行（48×512）"); ax[1].axis("off")
fig.tight_layout();

## 2. CTC — 「位置合わせなし」で学ぶ仕組み

### 困りごと
正解は「`沖縄300あ12-34`」という**文字列**だけです。でも画像のどこにどの文字があるかは教えていません。位置を教えずに、どうやって学習できるでしょうか？

### CTCの考え方
CRNNは、1行の画像を**128個の位置**（横に4ピクセルずつ）に区切り、**各位置で「どの文字か（または、何もない=blank）」の確率**を出します。

```
位置:   1   2   3   4   5   6   7   8   9 ...
出力:  blank 沖 沖 blank 縄 blank 3 3 blank ...    ← 各位置の「いちばん確からしい記号」
読む:  ① 連続する同じ記号を1つにまとめる   → 沖 縄 3 ...
       ② blank を取り除く                  →「沖縄3…」
```

- 同じ文字が**続く**とき（「22」など）は、**間にblankを入れて**区別します（「2, blank, 2」）。
- 学習では、「**正解の文字列になる並べ方**」すべての確率の合計が大きくなるように調整します（これが **CTC損失**）。**位置を教えなくて済む**のは、この「すべての並べ方」を数学的に足し合わせてくれるからです。

### 実際に見てみる
本実験で学習済みのモデルが、上の1行の画像を読むときの様子です。各位置で「blank以外でいちばん確からしい文字」を表示します。

In [ ]:
CK = ROOT / "outputs" / "ocr" / "crnn_all_cs3_s0_best.pt"
model = None
if CK.exists():
    model = CRNN().to(device); model.load_state_dict(torch.load(CK, map_location=device)); model.eval()
    with torch.no_grad():
        lp = model(x[None].to(device))                        # (128, 1, 語彙数): 各位置の「記号ごとの対数確率」
    p = lp.exp()[:, 0].cpu().numpy()
    best = p.argmax(1)
    text, conf = decode_greedy(lp)[0]
    print("読み取り結果:", text, "／ 正解:", s.label["text"].replace(" ", ""), "／ 一致:", text == s.label["text"].replace(" ", ""))
    fig, ax = plt.subplots(figsize=(11, 2.4))
    ax.imshow(x.permute(1, 2, 0).numpy(), extent=[0, 128, 0, 1], aspect="auto")
    for t, k in enumerate(best):
        if k != BLANK:                                        # blank以外の位置に、その文字と確率を表示
            ax.text(t + 0.5, 0.5, CHARS[k - 1], color="red", fontsize=13, ha="center", va="center", fontweight="bold")
            ax.text(t + 0.5, 0.1, f"{p[t, k]:.2f}", color="blue", fontsize=6, ha="center")
    ax.set(yticks=[], xlabel="位置（1位置 = 横4ピクセル。全部で128位置）"); fig.tight_layout()
else:
    print("学習済みのモデルがありません。ターミナルで  bash src/run_phase3.sh  を実行してください。")

**見るポイント（意外な発見）**
- 赤い文字は、**画像の文字の真上には出ていません。** 地名は左端に固まり、数字は画像の文字より**手前**に出ています。**文字の読み取りの「順番」は正しい**（左から右へ、沖→縄→3→1→L…）のに、**位置は対応していません**。
- **理由**: CRNNのRNNは、画像**全体の文脈**を見ながら（前からも後ろからも）読みます。そのため、「**文字が分かった時点**」で出力でき、その位置は文字の真上である必要がありません。CTCは「**順番**」さえ合えば良く、**位置の対応は強制しません**（これを「ピーキー（尖った）な出力」と呼びます）。
- つまり、CTCで学習したOCRは、**「どこに何があるか」ではなく「どの順で何が並ぶか」**を学びます。**文字の位置が必要な用途**（1文字ずつ枠で囲む、など）には、別の方法が向きます。
- 青い数字は、その位置での**確率**です。出力した位置では、ほぼ**1.00**（とても自信がある）になっています。ただし、一部の文字は**隣り合う複数の位置に確率が分散**し、確率が低く見えることがあります（あとの「確信度」の節で効いてきます）。
- 赤い文字が出ていない位置は、「blank（何もない）」を選んでいます。

## 3. 学習データ — 四隅の「ずれ」を混ぜる

OCRは、**検出器が出した四隅**で板を直した画像を読みます。検出器の四隅には、**1〜5px程度の誤差**があります（Phase2）。そこで学習のとき、**正解の四隅に、わざと誤差を足して**板を直します。

- **誤差の大きさ σ**（px）: 画像ごとに、0〜上限 の間でランダムに決める。
- 上限が**0**なら、**正解の四隅だけ**で学習（きれいすぎる）。上限が**大きすぎる**と、板が大きくゆがんで読めなくなる。
- ほど良い大きさは、**検出器の誤差に合わせる**のが基本です。

次のセルは、同じ1枚の画像を、誤差なし・小・大で直した板です。

In [ ]:
rng = np.random.default_rng(1)
fig, axes = plt.subplots(4, 1, figsize=(9, 4.6))
for ax, sigma in zip(axes, [0, 2, 3, 8]):
    c = jitter_corners(s.label["corners"], sigma, rng)
    ax.imshow(plate_to_tensor(rectify(s.image, c)).permute(1, 2, 0).numpy()); ax.axis("off")
    ax.set_title(f"四隅の誤差 σ={sigma}px で直した板（1行）", fontsize=9)
fig.tight_layout();

**見るポイント**: σ=3 でも、文字は**少し動く程度**ですが、σ=8 になると、**文字が欠けたり、大きくずれたり**します。学習に混ぜる誤差は、**検出器の実際の誤差**（1〜5px）に近い範囲が自然です。

## 4. 学習を短時間だけ体験する — CTCは「しばらく何も出さない」

CTCの学習には**独特の挙動**があります。短時間（約40秒）だけ学習して、様子を見ましょう。
- 最初のうちは、**ほとんど全部blank**（何も読まない）を出し続けます（lossは下がるが、文字は出ない）。
- ある時点から、**急に文字が出始め**ます。
本実験では、**40エポック × 4,500枚 = 180,000枚**（約10分）学習します。ここでは、その**ほんの入り口**（約1/12）だけです。

In [ ]:
EPOCHS_Q, N_Q, BATCH, LR = 10, 1280, 32, 1e-3             # ← 演習で変えてみる
torch.manual_seed(0)
mq = CRNN().to(device)
opt = torch.optim.AdamW(mq.parameters(), lr=LR, weight_decay=1e-4)
steps = EPOCHS_Q * (N_Q // BATCH)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=steps, pct_start=0.15)
ctc = nn.CTCLoss(blank=0, zero_infinity=True)                  # CTCの損失（lossの計算だけCPUで行う）
dl = DataLoader(OcrSynthDataset(EPOCHS_Q * N_Q), batch_size=BATCH, shuffle=False, num_workers=4, drop_last=True, collate_fn=collate)

spe, tot, n, t0 = N_Q // BATCH, 0.0, 0, time.time()
mq.train()
for step, (xb, yb, lens) in enumerate(dl, 1):
    lp = mq(xb.to(device))
    loss = ctc(lp.cpu(), yb, torch.full((xb.shape[0],), lp.shape[0], dtype=torch.long), lens)
    opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(mq.parameters(), 5.0); opt.step(); sched.step()
    tot += loss.item() * xb.shape[0]; n += xb.shape[0]
    if step % spe == 0:
        mq.eval()
        with torch.no_grad():
            txt = decode_greedy(mq(x[None].to(device)))[0][0]
        mq.train()
        print(f"エポック{step // spe:2d}: CTC loss {tot / n:6.3f} ／ 上の板の読み取り → 「{txt}」（正解: {s.label['text'].replace(' ', '')}）  ({time.time() - t0:.0f}秒)")
        tot, n = 0.0, 0

**見るポイント**: 最初の数エポックは「（空）」、そのあと**一部の文字だけ**が出始めます（たとえば、地名の漢字から）。**文字ごとに「できる時期」が違います**。
本実験の学習では、約20エポック目あたりで**一気に読めるようになり**、40エポックで落ち着きます。**学習が足りないと、一部の文字を読めずに終わる**ので、CTCは「十分に長く」学習する必要があります。

## 5. 本実験の結果 — Phase1・2と比べる

**本実験の結果**（40エポック × 4,500枚、標準設定は3回の平均±標準偏差）。評価は val の101枚目以降の各400枚（Phase1・2と同じ）です。
「**四隅の出どころ**」は、板を直すときの四隅を、**正解**から取るか、**Phase2 の検出器（ResNet18）の予測**から取るかです（後者が**システム全体**）。

| 難易度 | 手法 | 四隅の出どころ | 完全一致 | CER |
|---|---|---|---|---|
| easy | 古典的な検出＋テンプレートOCR（Phase1） | 古典的な検出 | 0.730 | 0.170 |
| easy | 古典的なテンプレートOCR | 正解 | 1.000 | 0.000 |
| easy | ResNet18の検出＋テンプレートOCR（Phase2） | 検出器の予測 | 0.938 ±0.005 | 0.009 |
| easy | **CRNN＋CTC（標準・3シード）** | 正解 | **0.979 ±0.011** | 0.002 ±0.001 |
| easy | **CRNN＋CTC（標準・3シード）** | 検出器の予測 | **0.976 ±0.015** | 0.002 ±0.002 |
| medium | 古典的な検出＋テンプレートOCR（Phase1） | 古典的な検出 | 0.122 | 0.506 |
| medium | 古典的なテンプレートOCR | 正解 | 0.718 | 0.032 |
| medium | ResNet18の検出＋テンプレートOCR（Phase2） | 検出器の予測 | 0.337 ±0.020 | 0.171 |
| medium | **CRNN＋CTC（標準・3シード）** | 正解 | **0.945 ±0.011** | 0.007 ±0.002 |
| medium | **CRNN＋CTC（標準・3シード）** | 検出器の予測 | **0.898 ±0.017** | 0.012 ±0.003 |
| hard | 古典的な検出＋テンプレートOCR（Phase1） | 古典的な検出 | 0.000 | 0.870 |
| hard | 古典的なテンプレートOCR | 正解 | 0.007 | 0.407 |
| hard | ResNet18の検出＋テンプレートOCR（Phase2） | 検出器の予測 | 0.001 ±0.001 | 0.656 |
| hard | **CRNN＋CTC（標準・3シード）** | 正解 | **0.199 ±0.008** | 0.207 ±0.007 |
| hard | **CRNN＋CTC（標準・3シード）** | 検出器の予測 | **0.081 ±0.004** | 0.319 ±0.007 |

### 何が分かるか
1. **学習するOCRは、劣化した画像に強い。** Medium の完全一致は、検出器の予測した四隅で **0.34（テンプレート）→ 0.90（CRNN）**。正解の四隅でも 0.72 → 0.95 です。**文字の見本を人間が書く必要がなく、ぼけ・ノイズ・汚れを含む例から学べる**ためです。
2. **検出器の誤差に強い。** テンプレートOCRは、正解の四隅 0.72 → 検出器の四隅 0.34 と**大きく落ちました**。CRNNは **0.95 → 0.90**。位置のずれを、学習のときに混ぜた（σ≦3px）おかげです。
3. **Easy は、ほぼ完璧**（0.98）。テンプレートOCRも正解の四隅なら100%でした。**差が出るのは、条件が悪くなってから**です。
4. **Hard は、まだ難しい。** 正解の四隅でも 0.20、検出器の四隅では 0.08。**画像から文字が実際に失われている**（次の節）ためで、モデルを大きくしても限界があります。

### フィールド別の正解率（標準・検出器の四隅）

| 難易度 | 地名 | 分類番号 | かな | 一連番号 |
|---|---|---|---|---|
| easy | 1.000 ±0.000 | 0.989 ±0.013 | 0.983 ±0.009 | 0.986 ±0.005 |
| medium | 0.992 ±0.003 | 0.939 ±0.021 | 0.939 ±0.026 | 0.966 ±0.004 |
| hard | 0.695 ±0.020 | 0.193 ±0.011 | 0.371 ±0.052 | 0.480 ±0.047 |

- Hard では、**地名（漢字5種類の識別）は0.70**ですが、**分類番号は0.19**、**かなは0.37**、**一連番号は0.48**。**数字の細部（7と1、6と8など）**が失われやすい、ということです。

### 実験: 四隅の誤差に対する、読み取りの強さ
Phase2 では、「**正解の四隅に誤差σを足す**」と、テンプレートOCRが**急に失敗する**ことを確かめました。同じ実験を、**CRNN**でもやってみます（正解の四隅に誤差を足し、Easy の60枚を読む）。

In [ ]:
if model is None:
    print("学習済みモデルがありません")
else:
    rng = np.random.default_rng(0)
    rows = []
    for sigma in [0, 1, 2, 3, 5, 8]:                           # ← 演習で変えてみる
        ok_t = ok_c = 0
        N = 60
        for i in range(N):
            smp = generate_sample(split_seed("val", "easy", 100 + i), "easy")
            c = np.asarray(smp.label["corners"]) + rng.normal(0, sigma, (4, 2))
            pl = rectify(smp.image, c)
            ok_t += exact_match(read_text(pl).text, smp.label["text"])                              # テンプレートOCR（Phase1）
            with torch.no_grad():
                pred = decode_greedy(model(plate_to_tensor(pl)[None].to(device)))[0][0]
            ok_c += pred == smp.label["text"].replace(" ", "")                                     # CRNN
        rows.append({"四隅の誤差 σ(px)": sigma, "テンプレートOCR": ok_t / N, "CRNN": ok_c / N})
    tol = pd.DataFrame(rows).set_index("四隅の誤差 σ(px)")
    tol.plot(marker="o", figsize=(6, 3.4), ylim=(0, 1.05), title="四隅の誤差と、読み取りの成功率（Easy）"); plt.grid(alpha=0.3)
tol

**見るポイント**: テンプレートOCRは、σが**2〜3pxで急落**します。CRNNは、**同じ誤差でもあまり落ちない**ことを確認しましょう（学習のとき、σ≦3px を混ぜたため、その範囲に強い）。ただし、**学習で見た範囲（3px）を超えると**、CRNNも落ちます（σ=8）。

## 6. 学習のしかたで、結果はどう変わるか

### 6-1. 四隅の誤差を、学習にどれだけ混ぜるか
学習のとき、四隅に足す誤差の**上限**を変えて比べました（1シードずつ）。

| 評価の難易度 | 学習時の四隅誤差の上限 | 四隅=正解 | 四隅=検出器の予測 |
|---|---|---|---|
| easy | 0（正解の四隅のみ） | 1.000 | 0.988 |
| easy | 3px（標準） | 0.990 | 0.992 |
| easy | 8px | 0.425 | 0.440 |
| medium | 0（正解の四隅のみ） | 0.985 | 0.770 |
| medium | 3px（標準） | 0.950 | 0.910 |
| medium | 8px | 0.405 | 0.390 |
| hard | 0（正解の四隅のみ） | 0.345 | 0.035 |
| hard | 3px（標準） | 0.195 | 0.085 |
| hard | 8px | 0.038 | 0.020 |

- **0（正解の四隅だけで学習）**: 正解の四隅で読むと**最高**（Medium 0.985、Hard 0.345）ですが、**検出器の四隅では Medium 0.77 に落ちる**。**きれいな条件で学習したAIは、現実のずれに弱い。**
- **3px（標準）**: 正解の四隅では少し下がる（Medium 0.95）が、**検出器の四隅では 0.91 と最も良い**。誤差を混ぜると、**位置ずれに強くなる代わりに、完璧な位置での精度は少し犠牲**になる（トレードオフ）。
- **8px**: **混ぜすぎ**。板が大きくゆがんで読めなくなり、**どちらでも崩壊**（Easy でも 0.43）。
- → **「検出器の実際の誤差に合わせて、ほどよく混ぜる」**ことが大切。これは**データ拡張**の考え方（前の演習）と同じです。ただし、**ラベルを壊さない**範囲に限ります（板を**左右反転**すると文字が鏡像になるので使えません。後の演習で確認）。

### 6-2. 学習データの難易度（分布シフト）

| 評価の難易度 | 学習データ | 四隅=正解 | 四隅=検出器の予測 |
|---|---|---|---|
| easy | Easy＋Medium＋Hard | 0.990 | 0.992 |
| easy | Easyだけ | 0.998 | 1.000 |
| medium | Easy＋Medium＋Hard | 0.950 | 0.910 |
| medium | Easyだけ | 0.130 | 0.100 |
| hard | Easy＋Medium＋Hard | 0.195 | 0.085 |
| hard | Easyだけ | 0.000 | 0.000 |

- **Easy だけで学習**すると、Easy は**むしろ最高**（0.998〜1.000）ですが、**Medium は 0.13、Hard は 0**。**学習した条件の外では、急に読めなくなる**（Phase2 の検出器と同じ）。
- **本番の条件（暗い、ぼける、汚れる）を、学習データに含める**ことが、設計の中心です。合成データなら、これを**自由に設計**できます。

## 7. 読み間違いを分析する

Hard や Medium で、**何を間違えているのか**を調べます。標準モデル（3シード、検出器の四隅）の誤りを、**文字単位**で集計します（文字数が同じ読み取りは、位置ごとに比べる。文字数が違う読み取りは、「脱落・余分」として別に数える）。

In [ ]:
frames = []
for sd in range(3):
    f = ROOT / "outputs" / "ocr" / f"crnn_all_cs3_s{sd}_val_pred.csv"
    if f.exists():
        frames.append(pd.read_csv(f).assign(seed=sd))
if not frames:
    print("評価結果がありません。bash src/run_phase3.sh を実行してください。")
else:
    ev = pd.concat(frames); ev["pred"] = ev["pred"].fillna("")
    rows = []
    for d in ["medium", "hard"]:
        x = ev[ev.difficulty == d]
        cnt, lenmis = Counter(), 0
        for gt, pr in zip(x["gt"], x["pred"]):
            if len(gt) == len(pr):
                cnt.update((a, b) for a, b in zip(gt, pr) if a != b)
            else:
                lenmis += 1
        print(f"【{d}】 文字数が違う読み取り（脱落・余分）: {lenmis / len(x):.1%} ／ よくある取り違え（正解→読み）:",
              [f"{a}→{b} ×{n}" for (a, b), n in cnt.most_common(8)])

**見るポイント**
- **Medium**: 間違いの種類はばらばらで、数も少ない。**小さな漢字（大↔沖、阪↔縄）**、**似た数字（6↔5、6↔8）**、**かな（は↔な、か↔た）**。
- **Hard**: **数字の取り違え**が圧倒的（7↔1、6↔8、6↔5、9↔5、8↔9）で、**文字数が違う読み取りが4分の1**。**ぼけ・ノイズ・低解像度で、文字の細部が失われている**ためです。**人間が見ても区別できない**場合が多く、**これは「AIの力不足」というより「画像に情報が残っていない」**問題です。
- → **読めない画像は、無理に答えさせず、「読み取り不可・要確認」として人に回す**のが正しい設計です。**そのための確信度**が、次の Phase4 のテーマです。

次のセルで、Medium の失敗例を実際に見ます。

In [ ]:
if frames:
    bad = ev[(ev.difficulty == "medium") & (~ev.exact) & (ev.seed == 0)].head(8)
    fig, axes = plt.subplots(2, 4, figsize=(14, 6.2))
    for ax, (_, r) in zip(axes.flat, bad.iterrows()):
        ax.imshow(Image.open(ROOT / "data" / "eval" / "val" / "medium" / "images" / r.image)); ax.axis("off")
        ax.set_title(f"正:{pretty(r['gt'])}\n読:{pretty(r['pred']) if len(r['pred']) >= 6 else r['pred']}", fontsize=8)
    fig.suptitle("Mediumの読み間違い（標準モデル・検出器の四隅）"); fig.tight_layout()

## 8. 確信度の予告 — 「どれくらい自信があるか」は使えるか

学習したAIは、**文字ごとの確率**を出します。この確率が、**正解と誤りを見分ける手がかり**になるでしょうか？ 各画像の「**最も自信のない文字の確率（conf_min）**」を、**正解した画像**と**誤った画像**で比べます。

In [ ]:
if frames:
    summ = ev.groupby(["difficulty", "exact"], sort=False).conf_min.mean().unstack().rename(columns={True: "正解した画像", False: "誤った画像"})
    display(summ.loc[["easy", "medium", "hard"]].round(3))
    rows = []
    for th in [0.5, 0.8, 0.9, 0.95]:
        a = ev[ev.conf_min >= th]
        rows.append({"しきい値": th, "自動処理率（全体）": round(len(a) / len(ev), 3), "自動処理した中の誤り率": round(1 - a.exact.mean(), 4) if len(a) else None})
    display(pd.DataFrame(rows).set_index("しきい値"))

**見るポイント**
- **正解した画像のほうが、確率が高い**（たとえば Easy で 0.82 vs 0.60）。確率は**手がかりにはなる**。
- ただし**差が小さい**。しきい値0.9にすると、**自動処理できるのは全体の約24%**で、その中にまだ**約2%の誤り**が残ります。**確率の取り方（最小値を取る、など）が、まだ粗い**ためです。
- → **確信度の作り方・しきい値の決め方**は、**Phase4**で工夫します（書式ルールによる補正、CTCの系列全体の確率、検証セットでのしきい値の決定、カバレッジ曲線）。

## 🧪 試してみよう

### 演習1: CTCの「立ち上がり」を観察する
**やること**: 「4. 学習を短時間だけ体験する」のセルの **`EPOCHS_Q = 10`** を **`20`**、**`30`** に書き換えて、そのセルを実行します（20は約80秒、30は約2分）。
- **見るポイント**: 上の板の読み取りが、**いつ、どの文字から**出始めますか？ 正解に近づきますか？
- **分かること**: CTCは**しばらく何も出さず、あとで一気に伸びる**。学習の長さが足りないと、**途中の中途半端な状態**で終わる。

### 演習2: 誤差の許容範囲を調べる
**やること**: 「実験: 四隅の誤差に対する、読み取りの強さ」のセルの **`[0, 1, 2, 3, 5, 8]`** を **`[0, 2, 4, 6, 8, 12]`** に書き換えて実行します。
- **見るポイント**: CRNNが急に落ちるのは、σがいくつのときですか？ **学習のとき混ぜた範囲（≦3px）**と関係していますか？
- **分かること**: AIは、**学習で経験した範囲の外**には弱い。

### 演習3: 左右反転した板を読ませる（ラベルを壊す理由）
**やること**: 下のセルを実行します。板を**左右反転**して、学習済みのAIに読ませます。
- **見るポイント**: 読み取りは正解と一致しますか？ 「沖縄」が鏡文字になった画像に、「沖縄」という**正解**を付けて学習するのは、**正しい**でしょうか？
- **分かること**: 反転した画像は、**もはや「正しい板」ではない**。**ラベル（文字列）と画像が食い違う**ので、データ拡張に使うと**学習が壊れます**（前の演習の、「ラベルを壊す拡張は使えない」）。

In [ ]:
if model is not None:
    mirrored = ImageOps.mirror(plate)                           # 左右反転
    with torch.no_grad():
        pred = decode_greedy(model(plate_to_tensor(mirrored)[None].to(device)))[0][0]
    print("正解:", s.label["text"].replace(" ", ""), "／ 反転した板の読み取り:", pred)
    show = Image.new("RGB", (384 * 2 + 10, 192), "white"); show.paste(plate, (0, 0)); show.paste(mirrored, (394, 0))
    plt.figure(figsize=(9, 2.4)); plt.imshow(show); plt.axis("off"); plt.title("左: 元の板 ／ 右: 左右反転した板（文字が鏡像になる）")

### 考えてみよう（コード不要）
1. **Medium の完全一致が0.34→0.90に上がった**のは、AIのどんな性質のおかげでしょう？ テンプレートOCRが苦手だったのは、何でしたか？
2. 「Hard は**画像に情報が残っていない**」とき、**AIをもっと大きく・長く学習**すれば読めるようになるでしょうか？ 読めるようになるとしたら、**何が起きている**ことになりますか（ヒント: 「推測」と「読み取り」の違い）？
3. 実際の駐車場システムで、**Hard のような画像**が来たとき、**誤って課金する**のと、**人に回す**のでは、どちらの損失が大きいでしょう？ それを設計に反映するには、何が必要ですか？

---

## ✅ まとめ

- **CRNN＋CTC**で、**例から読み方を学ぶOCR**を作った。**位置合わせなし**で文字列を学習できる（CTC）。学習は**しばらく何も出さず、あとで一気に伸びる**。
- **劣化に強い**: Medium の完全一致が、**テンプレートOCR 0.34 → CRNN 0.90**（検出器の四隅）。**検出器の位置誤差**にも強い（学習で四隅に誤差を混ぜたため）。
- **学習のしかたが結果を決める**: 誤差を混ぜない→現実のずれに弱い、混ぜすぎ→崩壊、**ほど良く**が正解。学習データの難易度も、AIの得意・不得意を決める（分布シフト）。
- **Hard は、画像に情報が残っておらず**、AIにも読めない（完全一致 0.08〜0.20）。**数字の細部が失われる**。
- **確率は手がかりになるが、粗い。** **読めないものは、人に回す**設計が必要 → 次のPhase。

➡ 次は **Phase4: ルール補正・確信度・「要確認」**。AIの読み取りを、**書式ルールで検査・補正**し、**自信がないものを人に回す**仕組みを作ります。カバレッジ（自動処理率）と誤り率のトレードオフを、グラフで見ます。